# Demo: orquestador multi-agente

**Pre-entrega 6 — Agustina Irisarri.** Demo del flujo de delegación.

Modo offline: agentes basados en reglas, búsqueda y datos simulados. Sin claves ni llamadas a un proveedor de IA. Resultado: 190 horas/mes de ahorro simulado. Se comprueban también refinamiento y parada.

## 1. Preparar
Ejecutar desde la raíz o desde `notebooks/`, con dependencias instaladas y el kernel del entorno `.venv`.

In [1]:
import sys
from pathlib import Path

root = Path.cwd() if Path('graph.py').exists() else Path.cwd().parent
assert (root / 'graph.py').exists(), 'Abrir desde el repositorio'
sys.path.insert(0, str(root))
from agents.analyst_agent import analyst_agent  # noqa: E402
from graph import build_graph, run  # noqa: E402
from main import DEMO_QUERY  # noqa: E402
from state import initial_state  # noqa: E402

print(DEMO_QUERY)

Investiga los pilotos de automatización RPA administrativos y calcula el ahorro mensual de tiempo.


## 2. Observar la delegación real
Cada línea se obtiene de un evento del grafo; no es una traza escrita a mano.

In [2]:
graph = build_graph()
for event in graph.stream(initial_state(DEMO_QUERY), {'recursion_limit': 100}):
    for node, update in event.items():
        print(node, '→', update.get('next_agent', 'aporte registrado'))
        if update.get('final_answer'):
            print('\n' + update['final_answer'])

supervisor → research
research → aporte registrado
supervisor → analyst
analyst → aporte registrado
supervisor → synthesis
synthesis → aporte registrado

Consulta: Investiga los pilotos de automatización RPA administrativos y calcula el ahorro mensual de tiempo.

Según los pilotos SIMULADOS recuperados, el tiempo mensual pasa de 256.67 a 66.67 horas. Ahorro: 190.00 horas/mes (74.03%).
El porcentaje es ponderado por tiempo inicial y volumen; no es un promedio simple.
Supuesto: volúmenes y tiempos constantes. No implica ahorro monetario ni evidencia real.
Fuentes:
- [rpa-02] Piloto administrativo: conciliación: simulado://piloto/conciliacion
- [rpa-03] Piloto administrativo: reportes: simulado://piloto/reportes
- [rpa-01] Piloto administrativo: facturas: simulado://piloto/facturas


## 3. Verificar el estado y la trazabilidad

In [3]:
result = run(DEMO_QUERY)
assert result['task_completed']
assert result['analysis']['saved_hours'] == 190
assert len(result['analysis']['source_ids']) == 3
for item in result['contributions']:
    print(f"{item['step']:02} | {item['agent']}: {item['detail']}")

01 | supervisor: research: Falta evidencia relevante
01 | research: search_corpus: 3 documentos
02 | supervisor: analyst: Falta análisis de la versión actual
02 | analyst: calculate_savings: validación y cálculo ponderado
03 | supervisor: synthesis: Rúbrica aprobada
03 | synthesis: Síntesis con fuentes y límites


## 4. Demostrar refinamiento
Inyección de fallo controlada: el primer análisis informa 999 horas. El supervisor detecta la inconsistencia y exige recalcular.

In [4]:
attempts = []
def incorrect_then_correct(state):
    attempts.append(1)
    update = analyst_agent(state)
    if len(attempts) == 1:
        update['analysis']['saved_hours'] = 999
    return update
refined = run(DEMO_QUERY, analyst_node=incorrect_then_correct)
assert refined['task_completed'] and len(attempts) == 2
assert refined['analysis']['saved_hours'] == 190
for item in refined['contributions']:
    print(item['agent'], '→', item['detail'])

supervisor → research: Falta evidencia relevante
research → search_corpus: 3 documentos
supervisor → analyst: Falta análisis de la versión actual
analyst → calculate_savings: validación y cálculo ponderado
supervisor → analyst: El cálculo no concilia con la evidencia
analyst → calculate_savings: validación y cálculo ponderado
supervisor → synthesis: Rúbrica aprobada
synthesis → Síntesis con fuentes y límites


## 5. Demostrar parada segura
Una consulta ajena al corpus no debe producir cifras inventadas ni un ciclo infinito.

In [5]:
incomplete = run('astrofísica cuántica galaxias', max_steps=3)
assert not incomplete['task_completed']
assert incomplete['steps'] == 4
print(incomplete['final_answer'])
print('Visitas al supervisor:', incomplete['steps'])

No se completó la tarea: Falta evidencia relevante
Visitas al supervisor: 4


## Comprobaciones
La ejecución verifica delegación, conservación de aportes, refinamiento y parada. Los resultados son simulados; no prueban beneficios reales. El modo LLM se configura en el README y requiere una API externa; este notebook no consume créditos ni afirma validar una llamada real.